# ArguScore-4B — judge server

Serves the fine-tuned debate scorer to the ARGUS backend over ngrok.

Run cells top to bottom. Cell 5 self-tests and must print `PASS` before the
backend is pointed at this session.

## Fixed defects (do not reintroduce)

| # | Defect | Symptom | Fix |
|---|--------|---------|-----|
| 1 | ChatML prompt (`<\|im_start\|>`) not in this Llama-3 vocab | Model ignored the supplied exchange and scored a debate it invented | `apply_chat_template()` |
| 2 | `eos_token_id` from `<\|im_end\|>` resolves to `None` | Generation never halted; ran full 2048 tokens (~180s) | `<\|eot_id\|>` (128009) |
| 3 | Reasoning block does not fit the budget | `Parse failed after N tok` on 6-turn rounds | Prefill `<think></think>` to skip |
| 4 | `fix_scores` clamped with `max(1.0, ...)` | Model could never emit 0, though rubric allows 0-2 | `max(0.0, ...)` |
| 5 | Endpoint returned errors with HTTP 200 | Backend saw a success with an unusable body | Return HTTP 500 |

Measured: **~187 tokens, ~12s** per 6-turn round at 16 tok/s on a T4.

In [ ]:
%%bash
pip install unsloth pyngrok fastapi uvicorn peft huggingface_hub -q
echo "done"

In [ ]:
# ── 1. Load ArguScore-4B ─────────────────────────────────────────────────────
from unsloth import FastLanguageModel
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient
import torch, json, re

secrets  = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")

print("Downloading ArguScore-4B v2...")
local = snapshot_download(
    repo_id="s7nket/ArguScore-4B",
    repo_type="model",
    token=HF_TOKEN,
)
ADAPTER_PATH = f"{local}/v2/adapter-final"
print(f"Adapter: {ADAPTER_PATH}")

ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_PATH,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)
FastLanguageModel.for_inference(ft_model)

# Guard against defect #2. Unsloth loads this adapter with a legacy tokenizer,
# and the ChatML tokens the original prompt used are absent from the vocab —
# convert_tokens_to_ids returns None and generation can never stop early.
EOT = ft_tokenizer.convert_tokens_to_ids("<|eot_id|>")
assert EOT is not None, "no <|eot_id|> in vocab — wrong tokenizer for this adapter"
assert ft_tokenizer.convert_tokens_to_ids("<|im_end|>") is None, \
    "vocab changed: <|im_end|> now resolves, revisit the prompt format"

print(f"ArguScore-4B ready.  eos=<|eot_id|>={EOT}")

In [ ]:
# ── 2. Rubric, parsing, scoring ──────────────────────────────────────────────
JUDGE_SYSTEM = """You are a strict debate judge. Analyze the debate exchange and return a JSON verdict.

Evaluate each side on:
- Evidence  (0-10): Facts, statistics, concrete examples cited
- Logic     (0-10): Structurally sound, no fallacies, clear reasoning chain
- Relevance (0-10): Directly and specifically addresses the debate topic
- total     = (evidence + logic + relevance) / 3

Return ONLY a single-line JSON object with no explanation before or after:
{"pro_scores":{"evidence":float,"logic":float,"relevance":float,"total":float},"con_scores":{"evidence":float,"logic":float,"relevance":float,"total":float},"round_winner":"pro or con or tie","reasoning":"one sentence only","winner_evidence":"strongest point the winner made","loser_weakness":"key failure of the losing side","fallacy_detected":null,"final_score_out_of_10":{"pro":float,"con":float}}"""


def extract_json(raw):
    """Strip reasoning and code fences, then recover the outermost JSON object."""
    raw = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()
    raw = raw.replace("```json", "").replace("```", "").strip()
    start = raw.find("{")
    if start == -1:
        return None
    try:
        return json.loads(raw[start:])
    except Exception:
        pass
    best, depth = None, 0
    for j, ch in enumerate(raw[start:], start):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                try:
                    best = json.loads(raw[start:j + 1])
                except Exception:
                    pass
    if best:
        return best
    # Last resort: close an unterminated string and any open braces.
    t = raw[start:]
    if t.count("{") > t.count("}"):
        if t.count('"') % 2 != 0:
            t = t[:t.rfind('"')] + '"'
        t += "}" * (t.count("{") - t.count("}"))
        try:
            return json.loads(t)
        except Exception:
            pass
    return None


def fix_scores(result):
    """
    Clamp to the rubric range and recompute totals.

    The floor is 0.0, not 1.0 (defect #4): the rubric defines 0-2 as valid for
    "pure assertion, nothing cited", so a floor of 1.0 biased the bottom of the
    scale on every label this model produced.
    """
    if result is None:
        return None
    for side in ("pro_scores", "con_scores"):
        s = result.get(side, {}) or {}
        e = s.get("evidence", 5.0)
        l = s.get("logic", 5.0)
        r = s.get("relevance", 5.0)
        t = s.get("total", 5.0)
        if t > 10:
            t = round((e + l + r) / 3, 1)
        s["evidence"]  = round(min(10.0, max(0.0, e)), 1)
        s["logic"]     = round(min(10.0, max(0.0, l)), 1)
        s["relevance"] = round(min(10.0, max(0.0, r)), 1)
        s["total"]     = round(min(10.0, max(0.0, t)), 1)
        result[side] = s
    pt = result["pro_scores"]["total"]
    ct = result["con_scores"]["total"]
    result["final_score_out_of_10"] = {"pro": pt, "con": ct}
    # Informational only. The backend recomputes the winner from totals using
    # TIE_BAND, which is just as well — this model's round_winner has been
    # observed contradicting its own scores.
    result["round_winner"] = "pro" if pt > ct + 0.3 else "con" if ct > pt + 0.3 else "tie"
    return result


# "skip"  -> prefill a closed think block, ~187 tok / ~12s   (production)
# "think" -> allow reasoning,             ~1483 tok / ~90s   (ablation)
#
# Kept switchable because the two modes disagreed on which side scored higher in
# a real round; which agrees better with human labels is unmeasured.
REASONING_MODE = "skip"
_PREFILL = "<think>\n\n</think>\n\n" if REASONING_MODE == "skip" else ""
_CAP     = 400 if REASONING_MODE == "skip" else 2048


def ft_judge_score(topic: str, exchange_text: str) -> dict:
    messages = [
        {"role": "system", "content": JUDGE_SYSTEM},
        {"role": "user", "content": f"Topic: {topic}\n\nExchange:\n{exchange_text}"},
    ]
    # apply_chat_template, not hand-built ChatML (defect #1). The template emits
    # <|begin_of_text|> itself, so add_special_tokens=False avoids a second BOS
    # shifting every position by one.
    prompt = ft_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    ) + _PREFILL
    inputs = ft_tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
    plen = inputs["input_ids"].shape[-1]

    import time
    t0 = time.time()
    with torch.no_grad():
        out = ft_model.generate(
            **inputs,
            max_new_tokens=_CAP,
            temperature=1e-6,
            do_sample=False,
            pad_token_id=ft_tokenizer.pad_token_id or EOT,
            eos_token_id=EOT,
        )
    gen = out.shape[-1] - plen
    body = ft_tokenizer.decode(out[0][plen:], skip_special_tokens=True)

    # Truncation is worth shouting about: a reasoning block cut off mid-sentence
    # leaves no JSON, and the only downstream symptom is a generic parse error.
    if gen >= _CAP:
        print(f"[ft] WARNING truncated at {gen}/{_CAP} tokens")

    result = fix_scores(extract_json(body))
    if result is None:
        raise ValueError(f"Parse failed after {gen} tok: {body[:200]}")

    print(f"[ft:{REASONING_MODE}] {gen} tok {time.time()-t0:.1f}s -> "
          f"pro={result['pro_scores']['total']} con={result['con_scores']['total']}")
    return result


print(f"ft_judge_score ready — mode={REASONING_MODE}, cap={_CAP}")

In [ ]:
# ── 3. Serve ─────────────────────────────────────────────────────────────────
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from pyngrok import ngrok
import uvicorn, threading

ngrok.set_auth_token(secrets.get_secret("NGROK_TOKEN"))

ft_app = FastAPI()


@ft_app.get("/health")
def health():
    return {"status": "online", "model": "ArguScore-4B", "mode": REASONING_MODE}


@ft_app.post("/ft-judge")
async def ft_judge_endpoint(data: dict):
    try:
        # Resolved at call time, not captured at definition time, so redefining
        # ft_judge_score in a later cell takes effect without restarting.
        return globals()["ft_judge_score"](data["topic"], data["exchange"])
    except Exception as e:
        # HTTP 500, not 200 (defect #5). Returning an error body with a success
        # status made the backend log a successful call with an unusable payload.
        return JSONResponse(status_code=500, content={"error": f"{type(e).__name__}: {e}"})


threading.Thread(
    target=lambda: uvicorn.run(ft_app, host="0.0.0.0", port=8002, log_level="error"),
    daemon=True,
).start()

ft_url = ngrok.connect(8002, "http").public_url
print("=" * 56)
print("ArguScore-4B SERVER LIVE")
print(f"\n  FT_JUDGE_URL={ft_url}\n")
print("=" * 56)

In [ ]:
# ── 4. Self-test — must print PASS before pointing the backend here ──────────
#
# Guards the two defects that produced confident, well-formed, WRONG output:
# scoring a hallucinated exchange, and silent truncation.
import json as _json

TOPIC = "Athens or Sparta: which was the better place to live?"

# Full production shape: 3 sub-rounds x 2 speakers.
REAL = """-- Sub-round 1: Opening --

PRO: Athens was the better place to live. Cleisthenes established the democratic assembly in 508 BCE, giving ordinary citizens a direct vote and trial by jury.

CON: Sparta was better. The archaeological record clearly shows historians agree Spartan society was more stable. Studies confirm this.

-- Sub-round 2: Counter --

PRO: My opponent cites nothing verifiable. Athenian juries numbered in the hundreds, drawn by lot, making bribery impractical.

CON: The Spartan Agoge produced the force that resisted Persia at Thermopylae in 480 BCE, proving institutional strength.

-- Sub-round 3: Justify --

PRO: Thermopylae was a defeat. Athens funded the fleet at Salamis that actually turned the war, and paid jurors so the poor could serve.

CON: Sparta granted women property rights and physical education unmatched in Athens, which materially improved life for half the population."""

r = ft_judge_score(TOPIC, REAL)
print(_json.dumps(r, indent=2)[:600])

blob = _json.dumps(r).lower()
ok = True

# The model must reference the supplied exchange. Under the old ChatML prompt it
# invented its own debate (Delian League, 404 BCE) and scored that instead.
if not any(k in blob for k in ("cleisthen", "508", "salamis", "agoge", "thermopylae", "jur")):
    print("FAIL: output does not reference the supplied exchange")
    ok = False

if any(k in blob for k in ("delian", "404", "peloponnes")):
    print("FAIL: output references content that was never supplied")
    ok = False

for side in ("pro_scores", "con_scores"):
    for k in ("evidence", "logic", "relevance", "total"):
        v = r[side][k]
        if not (0.0 <= v <= 10.0):
            print(f"FAIL: {side}.{k} = {v} outside 0-10")
            ok = False

# Determinism: greedy decoding must reproduce identical scores.
r2 = ft_judge_score(TOPIC, REAL)
if r2["pro_scores"] != r["pro_scores"] or r2["con_scores"] != r["con_scores"]:
    print("FAIL: not deterministic across identical calls")
    ok = False

print("\n" + "=" * 56)
print("PASS — safe to point the backend at this session" if ok else "FAIL — do not use")
print("=" * 56)
print(f"\nFT_JUDGE_URL={ft_url}")
print("Set that in backend/.env, then restart the backend.")